# HtmlRAG Deep Dive -- Why Metrics Collapse to Naive RAG

This notebook diagnoses why `htmlrag_style` produces **identical EM/F1 scores** to `naive_rag`.

We will prove two things:
1. **Same retrieval:** HtmlRAG uses the exact same FAISS search as Naive RAG
2. **Metrics collapse:** `_strip_html_for_metrics()` strips all HTML tags before computing EM/F1, making HTML context equivalent to plain text

Together, these two flaws guarantee that HtmlRAG and Naive RAG produce mathematically identical scores.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.corpus import load_corpus, load_hotpotqa
from src.embeddings import load_index, search
from src.retrieval import naive_rag, htmlrag_style, compute_em, compute_f1
from src.retrieval import _clean_html, _strip_html_for_metrics

# Load data
corpus = load_corpus(PROJECT_ROOT / "data" / "corpus.json")
index, page_ids = load_index(PROJECT_ROOT / "data")
qa_items = load_hotpotqa(split="train", n_samples=50)[:5]

print(f"Corpus: {len(corpus)} pages")
print(f"FAISS index: {index.ntotal} vectors")
print(f"QA items loaded: {len(qa_items)}")

## Step 1: Same Retrieval as Naive RAG

Both `naive_rag()` and `htmlrag_style()` call the **same `search()` function** with the **same FAISS index**.
The only difference is what happens to the retrieved pages:
- Naive RAG: concatenates `page["text"]` (plain text)
- HtmlRAG: concatenates `_clean_html(page["html"])` (cleaned HTML)

The **set of retrieved pages is identical**. Let's prove it.

In [ ]:
# Pick the first question
qa = qa_items[0]
question = qa["question"]
answer = qa["answer"]

print(f"Question: {question}")
print(f"Gold answer: {answer}")
print()

# Both systems use the same search
retrieved = search(question, index, page_ids, corpus, k=5)
retrieved_titles = [r["title"] for r in retrieved]

print("Retrieved pages (shared by both Naive RAG and HtmlRAG):")
for i, title in enumerate(retrieved_titles):
    print(f"  {i+1}. {title}")

print()
print("PROOF: Both naive_rag() and htmlrag_style() call search() with the")
print("same index, same page_ids, same k=5. The retrieval is IDENTICAL.")
print("Differentiation is ONLY in context formatting (plain text vs HTML).")

## Step 2: HTML Cleaning -- What `_clean_html()` Does

HtmlRAG passes each retrieved page's HTML through `_clean_html()`, which:
- Removes `<script>`, `<style>`, and `<nav>` elements
- Preserves structural tags like `<h1>`, `<p>`, `<table>`, `<li>`

Let's look at a page in all three forms: plain text, raw HTML, and cleaned HTML.

In [ ]:
# First retrieved page
page = retrieved[0]

print("=" * 60)
print("PLAIN TEXT (page['text'][:300]):")
print("=" * 60)
print(page["text"][:300])
print()

print("=" * 60)
print("RAW HTML (page['html'][:300]):")
print("=" * 60)
print(page["html"][:300])
print()

print("=" * 60)
print("CLEANED HTML (_clean_html(page['html'])[:300]):")
print("=" * 60)
print(_clean_html(page["html"])[:300])
print()
print("Note: _clean_html preserves structural tags (h1, p, table, li)")
print("but removes script/style/nav. The content itself is unchanged.")

## Step 3: The Metrics Collapse -- `_strip_html_for_metrics()`

This is the **critical flaw**. Both `compute_em()` and `compute_f1()` call `_strip_html_for_metrics()` on the prediction context before comparing with the gold answer.

This function strips ALL HTML tags using BeautifulSoup's `get_text()`. After stripping:
- The cleaned HTML context becomes plain text
- This plain text is (nearly) identical to the Naive RAG plain text context

**Result:** EM and F1 are computed on identical text, producing identical scores.

In [ ]:
# Get contexts from both systems
naive_context = naive_rag(question, index, corpus, k=5)
html_context = htmlrag_style(question, index, corpus, k=5)

print(f"Naive context length: {len(naive_context)} chars")
print(f"HTML context length:  {len(html_context)} chars")
print(f"Difference: {len(html_context) - len(naive_context)} chars (HTML tags add overhead)")
print()

# Strip HTML for metrics
stripped_html = _strip_html_for_metrics(html_context)
print(f"Stripped HTML length: {len(stripped_html)} chars")
print()

# Side-by-side comparison
print("=" * 60)
print("NAIVE CONTEXT (first 200 chars):")
print(naive_context[:200])
print()
print("STRIPPED HTML CONTEXT (first 200 chars):")
print(stripped_html[:200])
print("=" * 60)
print()

# Key diagnostic: are they the same after stripping?
# Note: they may not be char-for-char identical due to whitespace normalization,
# but the TOKEN content used by compute_f1 is the same
naive_tokens = set(naive_context.lower().split())
stripped_tokens = set(stripped_html.lower().split())
token_diff = naive_tokens.symmetric_difference(stripped_tokens)
print(f"Token set symmetric difference: {len(token_diff)} tokens differ")
if token_diff:
    print(f"  (minor differences from whitespace normalization)")
print()

# Compute EM and F1 for both
naive_em = compute_em(naive_context, answer)
html_em = compute_em(html_context, answer)
naive_f1 = compute_f1(naive_context, answer)
html_f1 = compute_f1(html_context, answer)

print(f"Naive RAG  -> EM={naive_em}, F1={naive_f1:.6f}")
print(f"HtmlRAG    -> EM={html_em}, F1={html_f1:.6f}")
print(f"EM match: {naive_em == html_em}")
print(f"F1 match: {naive_f1 == html_f1}")
print()
print("CONCLUSION: _strip_html_for_metrics() collapses HTML context to plain text")
print("BEFORE EM/F1 computation. The scores are MATHEMATICALLY GUARANTEED to match.")

## Step 4: Side-by-Side Comparison Across Questions

Let's run both systems on all 5 QA items and confirm the metrics are identical for every single question.

In [ ]:
comparison = []

for i, qa in enumerate(qa_items):
    q = qa["question"]
    a = qa["answer"]

    ctx_naive = naive_rag(q, index, corpus, k=5)
    ctx_html = htmlrag_style(q, index, corpus, k=5)

    n_em = compute_em(ctx_naive, a)
    h_em = compute_em(ctx_html, a)
    n_f1 = compute_f1(ctx_naive, a)
    h_f1 = compute_f1(ctx_html, a)

    comparison.append({
        "qid": i, "naive_em": n_em, "html_em": h_em,
        "naive_f1": n_f1, "html_f1": h_f1,
        "em_match": n_em == h_em, "f1_match": n_f1 == h_f1
    })

# Print comparison table
print(f"{'QID':<5} {'N_EM':<6} {'H_EM':<6} {'N_F1':<10} {'H_F1':<10} {'EM?':<5} {'F1?'}")
print("-" * 52)
for c in comparison:
    print(f"{c['qid']:<5} {c['naive_em']:<6.0f} {c['html_em']:<6.0f} "
          f"{c['naive_f1']:<10.6f} {c['html_f1']:<10.6f} "
          f"{str(c['em_match']):<5} {c['f1_match']}")

all_em_match = all(c["em_match"] for c in comparison)
all_f1_match = all(c["f1_match"] for c in comparison)
print(f"\nAll EM scores match: {all_em_match}")
print(f"All F1 scores match: {all_f1_match}")

# Scatter plot: naive_f1 vs html_f1 (should fall on y=x line)
fig, ax = plt.subplots(figsize=(6, 6))
naive_f1s = [c["naive_f1"] for c in comparison]
html_f1s = [c["html_f1"] for c in comparison]

ax.scatter(naive_f1s, html_f1s, s=100, zorder=5, color="coral", edgecolors="black")

# y=x reference line
lims = [0, max(max(naive_f1s), max(html_f1s)) * 1.1 + 0.001]
ax.plot(lims, lims, "k--", alpha=0.5, label="y = x (identical)")

ax.set_xlabel("Naive RAG F1")
ax.set_ylabel("HtmlRAG F1")
ax.set_title("Naive RAG vs HtmlRAG F1 Scores\n(all points ON the y=x line = identical)")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Diagnosis

### Flaw 1: Shared Retrieval

`htmlrag_style()` uses the **exact same** `search()` function with the **same FAISS index** as `naive_rag()`. The only difference is that HtmlRAG formats the context as cleaned HTML (via `_clean_html()`) instead of plain text. The **set of retrieved pages is identical** -- not just similar, but the exact same pages in the exact same order.

This means HtmlRAG has **no opportunity to retrieve different or better pages** than Naive RAG. The HTML formatting is cosmetic only.

### Flaw 2: Metrics Collapse via `_strip_html_for_metrics()`

Both `compute_em()` and `compute_f1()` call `_strip_html_for_metrics()` on the prediction context before comparing with the gold answer. This function uses BeautifulSoup to strip ALL HTML tags, converting:

```
<p>France is a country in <b>Europe</b>.</p>
```
to:
```
France is a country in Europe.
```

After stripping, the HTML context becomes plain text that is functionally identical to the Naive RAG context. Since EM and F1 are computed on the stripped text, the scores are **mathematically guaranteed** to be the same.

### What HtmlRAG SHOULD Do Differently

For HtmlRAG to produce meaningfully different results, it would need to:

1. **Use HTML structure for retrieval:** Weight heading matches higher, use table structure for tabular queries, etc.
2. **Use structure-aware metrics:** Evaluate whether the answer appears in semantically relevant sections (headings, infoboxes) vs buried in body text.
3. **Apply HTML-specific re-ranking:** After initial FAISS retrieval, re-rank pages based on how well their HTML structure matches the query type.

Without these changes, HtmlRAG is just Naive RAG with extra HTML tags that get stripped before evaluation.